In [ ]:
!pip install ydata-profiling

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from ydata_profiling import ProfileReport
import warnings
warnings.filterwarnings('ignore')
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier


/tmp/ipykernel_2692/1182810170.py:5: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


In [4]:
df = pd.read_csv('/content/heart.csv')

In [5]:
df.sample(5)

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
842,43,F,ASY,132,341,1,LVH,136,Y,3.0,Flat,1
350,53,M,ASY,120,0,1,Normal,120,N,0.0,Flat,1
825,59,M,TA,170,288,0,LVH,159,N,0.2,Flat,1
484,65,M,ASY,134,0,0,Normal,112,Y,1.1,Flat,1
763,58,M,NAP,132,224,0,LVH,173,N,3.2,Up,1


In [6]:
profile = ProfileReport(df, title="Dataset Profile", explorative=True)
profile.to_file("report1.html")

Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]


100%|██████████| 12/12 [00:00<00:00, 17.17it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 918 entries, 0 to 917
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Age             918 non-null    int64  
 1   Sex             918 non-null    object 
 2   ChestPainType   918 non-null    object 
 3   RestingBP       918 non-null    int64  
 4   Cholesterol     918 non-null    int64  
 5   FastingBS       918 non-null    int64  
 6   RestingECG      918 non-null    object 
 7   MaxHR           918 non-null    int64  
 8   ExerciseAngina  918 non-null    object 
 9   Oldpeak         918 non-null    float64
 10  ST_Slope        918 non-null    object 
 11  HeartDisease    918 non-null    int64  
dtypes: float64(1), int64(6), object(5)
memory usage: 86.2+ KB


In [8]:
df.duplicated().sum()

np.int64(0)

In [9]:
df['Cholesterol'].value_counts()

,count
Cholesterol,
0,172
254,11
220,10
223,10
204,9
...,...
353,1
278,1
157,1


In [10]:
#one cant have a cholestrol of 0

In [11]:
c = df.loc[df['Cholesterol'] != 0 ,'Cholesterol'].mean()

In [12]:
df['Cholesterol'] = df['Cholesterol'].replace(0,c)

In [13]:
#also in resting bp one cant have a resting bp of 0 and it acts as an outlier

In [14]:
df['RestingBP'].value_counts().get(0)

np.int64(1)

In [15]:
c = df.loc[df['RestingBP'] != 0 ,'RestingBP'].mean()

In [16]:
df['RestingBP'] = df['RestingBP'].replace(0,c)

In [17]:
#high corr with chestpain type and Stslope

In [18]:
df.sample(10)

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
20,43,F,TA,100.0,223.000000,0,Normal,142,N,0.0,Up,0
747,42,M,ASY,140.0,226.000000,0,Normal,178,N,0.0,Up,0
360,62,M,NAP,160.0,244.635389,0,Normal,72,Y,0.0,Flat,1
214,47,M,ASY,150.0,226.000000,0,Normal,98,Y,1.5,Flat,1
660,58,M,NAP,140.0,211.000000,1,LVH,165,N,0.0,Up,0
167,48,M,ATA,140.0,238.000000,0,Normal,118,N,0.0,Up,0
663,66,M,ATA,160.0,246.000000,0,Normal,120,Y,0.0,Flat,1
635,67,M,ASY,120.0,229.000000,0,LVH,129,Y,2.6,Flat,1
319,68,M,ASY,145.0,244.635389,1,Normal,136,N,1.8,Up,1
820,52,M,TA,152.0,298.000000,1,Normal,178,N,1.2,Flat,0


In [19]:
df_enc = pd.get_dummies(df,drop_first=True)

In [20]:
df_enc.sample(10)

,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak,HeartDisease,Sex_M,ChestPainType_ATA,ChestPainType_NAP,ChestPainType_TA,RestingECG_Normal,RestingECG_ST,ExerciseAngina_Y,ST_Slope_Flat,ST_Slope_Up
190,46,180.0,280.0,0,120,0.0,0,True,False,False,False,False,True,False,False,True
721,51,100.0,222.0,0,143,1.2,0,True,False,True,False,True,False,True,True,False
443,60,130.0,186.0,1,140,0.5,1,True,False,False,False,False,True,True,True,False
134,61,130.0,294.0,0,120,1.0,0,False,False,False,False,False,True,True,True,False
710,47,110.0,275.0,0,118,1.0,1,True,False,False,False,False,False,True,True,False
796,56,134.0,409.0,0,150,1.9,1,False,False,False,False,False,False,True,True,False
15,54,120.0,273.0,0,150,1.5,0,False,True,False,False,True,False,False,True,False
280,60,120.0,246.0,0,135,0.0,0,True,False,True,False,False,False,False,False,True
625,59,135.0,234.0,0,161,0.5,0,True,False,False,False,True,False,False,True,False
743,52,134.0,201.0,0,158,0.8,0,True,True,False,False,True,False,False,False,True


In [21]:
df_enc.astype(int)

,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak,HeartDisease,Sex_M,ChestPainType_ATA,ChestPainType_NAP,ChestPainType_TA,RestingECG_Normal,RestingECG_ST,ExerciseAngina_Y,ST_Slope_Flat,ST_Slope_Up
0,40,140,289,0,172,0,0,1,1,0,0,1,0,0,0,1
1,49,160,180,0,156,1,1,0,0,1,0,1,0,0,1,0
2,37,130,283,0,98,0,0,1,1,0,0,0,1,0,0,1
3,48,138,214,0,108,1,1,0,0,0,0,1,0,1,1,0
4,54,150,195,0,122,0,0,1,0,1,0,1,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
913,45,110,264,0,132,1,1,1,0,0,1,1,0,0,1,0
914,68,144,193,1,141,3,1,1,0,0,0,1,0,0,1,0
915,57,130,131,0,115,1,1,1,0,0,0,1,0,1,1,0
916,57,130,236,0,174,0,1,0,1,0,0,0,0,0,1,0


In [22]:
df_enc.columns

Index(['Age', 'RestingBP', 'Cholesterol', 'FastingBS', 'MaxHR', 'Oldpeak',
       'HeartDisease', 'Sex_M', 'ChestPainType_ATA', 'ChestPainType_NAP',
       'ChestPainType_TA', 'RestingECG_Normal', 'RestingECG_ST',
       'ExerciseAngina_Y', 'ST_Slope_Flat', 'ST_Slope_Up'],
      dtype='object')

In [23]:
sc = StandardScaler()
unscaled = ['Age', 'RestingBP', 'Cholesterol','MaxHR', 'Oldpeak']
df_enc[unscaled] = sc.fit_transform(df_enc[unscaled])

In [24]:
df_enc.sample(10)

,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak,HeartDisease,Sex_M,ChestPainType_ATA,ChestPainType_NAP,ChestPainType_TA,RestingECG_Normal,RestingECG_ST,ExerciseAngina_Y,ST_Slope_Flat,ST_Slope_Up
498,1.430829,1.527192,2.615261,1,-0.267596,-0.832432,1,True,False,False,False,False,True,True,True,False
555,0.476173,0.971022,-0.481064,0,-0.739174,-0.832432,1,True,False,True,False,False,True,True,True,False
653,-1.220994,-0.141317,-1.212922,0,0.518368,-0.832432,0,True,False,True,False,True,False,False,False,True
100,1.218683,-0.141317,0.569810,0,-0.857069,0.105664,1,True,False,False,False,False,True,True,True,False
735,-0.478484,-0.697486,-1.062797,0,0.086088,1.043759,1,True,False,True,False,True,False,False,True,False
596,0.370100,-0.586252,0.363389,0,-1.446542,-0.832432,1,True,False,False,False,False,False,False,True,False
471,1.536902,0.303619,0.000000,0,-0.267596,1.981855,1,True,False,False,False,True,False,True,True,False
40,0.051881,0.971022,-0.274642,0,-0.267596,-0.832432,0,False,True,False,False,True,False,False,False,True
26,-0.054192,-0.475018,0.288326,0,-0.974963,1.981855,0,True,False,False,False,False,True,True,True,False
791,-0.266338,0.414853,1.001419,0,-0.581981,3.107570,1,True,False,False,False,True,False,True,True,False


In [25]:
round(df_enc.describe())

,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak,HeartDisease
count,918.0,918.0,918.0,918.0,918.0,918.0,918.0
mean,-0.0,-0.0,-0.0,0.0,0.0,0.0,1.0
std,1.0,1.0,1.0,0.0,1.0,1.0,0.0
min,-3.0,-3.0,-3.0,0.0,-3.0,-3.0,0.0
25%,-1.0,-1.0,-1.0,0.0,-1.0,-1.0,0.0
50%,0.0,-0.0,0.0,0.0,0.0,-0.0,1.0
75%,1.0,0.0,0.0,0.0,1.0,1.0,1.0
max,2.0,4.0,7.0,1.0,3.0,5.0,1.0


In [26]:
models = {
    "Logistic Regression": LogisticRegression(),
    "KNN": KNeighborsClassifier(),
    "Naive Bayes": GaussianNB(),
    "Decision Tree": DecisionTreeClassifier(),
    "SVM (RBF Kernel)": SVC(probability=True)
}

In [27]:
X = df_enc.drop('HeartDisease', axis=1)
y = df_enc['HeartDisease']

In [28]:
X_train, X_test, y_train, y_test = train_test_split( X, y, stratify=y, test_size=0.2, random_state=42)

In [29]:
X_train_scaled = sc.fit_transform(X_train)
X_test_scaled = sc.transform(X_test)

In [30]:
results=[]

In [31]:
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    results.append({
        'Model': name,
        'Accuracy': round(acc, 4),
        'F1 Score': round(f1, 4)
    })


In [32]:
results

[{'Model': 'Logistic Regression', 'Accuracy': 0.8913, 'F1 Score': 0.9029},
 {'Model': 'KNN', 'Accuracy': 0.8913, 'F1 Score': 0.9029},
 {'Model': 'Naive Bayes', 'Accuracy': 0.875, 'F1 Score': 0.8844},
 {'Model': 'Decision Tree', 'Accuracy': 0.7663, 'F1 Score': 0.7882},
 {'Model': 'SVM (RBF Kernel)', 'Accuracy': 0.8587, 'F1 Score': 0.875}]

In [35]:
import joblib
joblib.dump(models['KNN'],  'knn_heart.pkl')
joblib.dump(sc,  'scaler.pkl')
joblib.dump(X.columns.tolist(),  'cols.pkl')

['cols.pkl']